In [ ]:
import random
import numpy as np
from pathlib import Path
from the_well.data import WellDataset

import torch
from torch.utils.data import DataLoader

from modules import *
from losses import *
from datasets import HelmholtzDataset

In [ ]:
SEED         = 42
EPOCHS       = 100
BATCH_SIZE   = 8
LR           = 1e-3

DATASET_PATH = "/mnt/storage_C1/BILL_pino/the_well/datasets/helmholtz_staircase"
OUTPUT_DIR   = "/mnt/storage_C1/BILL_pino/the_well/models"

MODES1       = 16   # Modos de Fourier na primeira dimensão espacial (x)
MODES2       = 16   # Modos de Fourier na segunda dimensão espacial (y)
WIDTH        = 32   # Número de canais internos (largura do modelo)
DEPTH        = 4    # Quantidade de camadas de Fourier
PROJ_DIM     = 128  # Dimensão da MLP de projeção para a saída

In [3]:
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Dispositivo: {device}")
print(f"Torch: {torch.__version__}  |  Torchvision: {__import__('torchvision').__version__}")

Dispositivo: cuda
Torch: 2.11.0+cu128  |  Torchvision: 0.26.0+cu128


In [ ]:
train_dataset = WellDataset(
    path=DATASET_PATH,
    well_split_name="train"
)

validation_dataset = WellDataset(
    path=DATASET_PATH,
    well_split_name="valid"
)

train_ds = HelmholtzDataset(train_dataset)
val_ds = HelmholtzDataset(validation_dataset)

train_loader = DataLoader(
    train_ds,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=16,
    pin_memory=True,
    persistent_workers=True,
    prefetch_factor=4,
)

val_loader = DataLoader(
    val_ds,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=16,
    pin_memory=True,
    persistent_workers=True,
    prefetch_factor=4,
)

In [ ]:
x, y = train_ds[0]
in_dim, out_dim = x.shape[-1], y.shape[-1] 

model = FNO2d(
    modes1=MODES1,
    modes2=MODES2,
    width=WIDTH,
    in_dim=in_dim,
    out_dim=out_dim,
    depth=DEPTH,
    proj_dim=PROJ_DIM
).cuda()
model = torch.compile(model)

optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=LR
)

criterion = relative_l2_loss

/home/al.igor.zwirtes/Documentos/AV2_ML2_IMPA/.env/lib/python3.12/site-packages/the_well/data/datasets.py:736: RuntimeWarning: Only axis-aligned boundary fully supported. Boundary for axis counted as `open` or `periodic` if any part of it is and `wall` otherwise.If this does not fit your desired usecase, set `boundary_return_type=None`.
  warnings.warn(


In [ ]:
history = train_model(
    model=model,
    train_loader=train_loader,
    val_loader=val_loader,
    optimizer=optimizer,
    criterion=criterion,
    epochs=EPOCHS,
    checkpoint_dir=OUTPUT_DIR
)

Epoch 1/100: 100%|██████████| 2548/2548 [31:10<00:00,  1.36it/s, loss=0.094741]  
/home/al.igor.zwirtes/Documentos/AV2_ML2_IMPA/.env/lib/python3.12/site-packages/the_well/data/datasets.py:736: RuntimeWarning: Only axis-aligned boundary fully supported. Boundary for axis counted as `open` or `periodic` if any part of it is and `wall` otherwise.If this does not fit your desired usecase, set `boundary_return_type=None`.
  warnings.warn(
/home/al.igor.zwirtes/Documentos/AV2_ML2_IMPA/.env/lib/python3.12/site-packages/the_well/data/datasets.py:736: RuntimeWarning: Only axis-aligned boundary fully supported. Boundary for axis counted as `open` or `periodic` if any part of it is and `wall` otherwise.If this does not fit your desired usecase, set `boundary_return_type=None`.
  warnings.warn(
/home/al.igor.zwirtes/Documentos/AV2_ML2_IMPA/.env/lib/python3.12/site-packages/the_well/data/datasets.py:736: RuntimeWarning: Only axis-aligned boundary fully supported. Boundary for axis counted as `open`

Epoch 001 | train = 0.094741 | val = 0.032026 | best_val = 0.032026


Epoch 2/100: 100%|██████████| 2548/2548 [31:01<00:00,  1.37it/s, loss=0.034687]  


Epoch 002 | train = 0.034687 | val = 0.031871 | best_val = 0.031871


Epoch 3/100: 100%|██████████| 2548/2548 [14:07<00:00,  3.01it/s, loss=0.030965]


...

Epoch 4/100: 100%|██████████| 2548/2548 [13:33<00:00,  3.13it/s, loss=0.028910]


Epoch 004 | train = 0.028910 | val = 0.029441 | best_val = 0.029441


Epoch 5/100: 100%|██████████| 2548/2548 [12:46<00:00,  3.33it/s, loss=0.026451]


Epoch 005 | train = 0.026451 | val = 0.023720 | best_val = 0.023720


Epoch 6/100: 100%|██████████| 2548/2548 [12:40<00:00,  3.35it/s, loss=0.024713]
